# Regenerate CARL evals after the speculative-decoding fix (GPU)

The original eval artifacts (`failure_cases_results.json`, `ablation_live_results.json`, `cross_model_results.json`) are **unrecoverable** and were produced while the speculative-decoding regression `9a5a4c9` was live (fixed in `2e11321`). See `docs/eval/spec_decode_contamination.md`.

Since there is no recoverable baseline, this notebook regenerates **both sides on GPU**:
* **before/** = pre-fix commit `f23d6ff` (contains the bug)
* **after/**  = post-fix commit `2e11321`

then diffs them and flags `spec_k > 0` impact, winner flips, and ranking changes.

**Prerequisites:** a GPU runtime, and both commits + this `scripts/eval/regen/` folder pushed to `origin`.

**Runtime:** order of hours on a single GPU (dominated by `ablation_live` and the `memory_pressure` scenario). Leave it running.

In [ ]:
# 1) Confirm GPU. The eval code treats CPU as smoke-only; do not proceed on CPU.
import torch, subprocess
assert torch.cuda.is_available(), 'No CUDA. Switch the Colab runtime to a GPU before running.'
print('GPU:', torch.cuda.get_device_name(0))
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
# 2) Clone the repo. Replace REPO_URL if your remote differs.
import os
REPO_URL = 'https://github.com/vsvidhun06-blip/mini-vllm.git'
if not os.path.isdir('/content/mini-vllm'):
    !git clone $REPO_URL /content/mini-vllm
%cd /content/mini-vllm
!git fetch --all --quiet && git log --oneline -3

In [ ]:
# 3) Install deps. (requirements.txt is UTF-16; re-encode if pip complains.)
!pip install -q -r requirements.txt || (iconv -f UTF-16 -t UTF-8 requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt)
# TinyLlama is downloaded on first model load; cross_model.py pulls additional
# checkpoints. The first run will spend time downloading.

In [ ]:
# 4) Copy the regen tools OUTSIDE the working tree so they survive git checkout.
!mkdir -p /content/regen_tools /content/out/before /content/out/after
!cp scripts/eval/regen/run_phase.sh scripts/eval/regen/compare_results.py /content/regen_tools/

# Commits to compare. Stable hashes; both must exist on origin.
BEFORE = 'f23d6ff'   # pre-fix (bug present)
AFTER  = '2e11321'   # post-fix

# Seeds. Empty string => each script's NATIVE default = the ORIGINAL config
# (failure_cases/cross_model = 42,43,44 ; ablation_live = 42..51, N=10).
# The task asked for seeds 42-44; to force that everywhere set SEEDS='42,43,44',
# but note that deviates from ablation_live's original N=10. Keep BEFORE and
# AFTER identical either way.
SEEDS = ''   # '' = original config; or '42,43,44'
LIMIT = ''   # '' = original requests-per-run
os.environ.update(BEFORE=BEFORE, AFTER=AFTER, SEEDS=SEEDS, LIMIT=LIMIT)
print('before=%s after=%s seeds=%r limit=%r' % (BEFORE, AFTER, SEEDS or 'native', LIMIT or 'native'))

In [ ]:
# 5) PRE-FIX phase (the contaminated baseline). Hours; dominated by memory_pressure.
!bash /content/regen_tools/run_phase.sh "$BEFORE" /content/out/before "$SEEDS" "$LIMIT"

In [ ]:
# 6) POST-FIX phase.
!bash /content/regen_tools/run_phase.sh "$AFTER" /content/out/after "$SEEDS" "$LIMIT"

In [ ]:
# 7) Compare and render the report (also saved to /content/out/comparison_report.md).
!python /content/regen_tools/compare_results.py --before /content/out/before --after /content/out/after --out /content/out/comparison_report.md
print('\n--- acceptance probe: BEFORE ---'); print(open('/content/out/before/acceptance_probe.txt').read())
print('--- acceptance probe: AFTER ---'); print(open('/content/out/after/acceptance_probe.txt').read())

In [ ]:
# 8) (optional) Render the markdown report inline, and download the artifacts.
from IPython.display import Markdown, display
display(Markdown(open('/content/out/comparison_report.md').read()))
# from google.colab import files; files.download('/content/out/comparison_report.md')